# Programming Assignment 1
## From Tableau to Decision: Simplex, Duality, and Sensitivity Analysis

**Name:** Basil George

This guided notebook follows one production-planning problem from its initial simplex tableau to a managerial capacity decision. Work from top to bottom. Run a code cell with **Shift+Enter** and write answers in cells labeled **YOUR RESPONSE**.

## New to Jupyter? Start here

A notebook is a sequence of cells. Click a cell once to select it.

- A **code cell** contains Python and produces output below it. Run it with **Shift+Enter**.
- A **YOUR CHOICE** code cell asks you to replace the empty text between quotation marks. For example, change `entering_0 = ""` to `entering_0 = "x3"` if `x3` is your choice. Keep the quotation marks.
- A **YOUR RESPONSE** cell is a writing area. Double-click it, replace the bold placeholder text with your work, and press **Shift+Enter** to display the formatted answer.
- A cell labeled **RUN** is supplied for you. Run it, but do not edit it.
- The number beside a code cell records when it was run. If cells were run out of order, use **Kernel > Restart Kernel and Run All Cells**.

Save frequently. A red box or traceback means that a code cell encountered an error; consult `TROUBLESHOOTING.md` before proceeding.

### Academic integrity and permitted help

Your course's academic-integrity policy governs this assignment. Unless your instructor states otherwise, you may use course notes, the textbook, official Python/Jupyter documentation, and instructor or teaching-assistant help. You may discuss general concepts with classmates, but your pivot choices, calculations, explanations, and decision memo must be your own. Do not share completed files, copy another student's responses, use a generated solution as your submitted explanation, or alter instructor-provided checking code.

## Learning objectives

After completing the assignment, you should be able to:

1. Trace the simplex method as a sequence of adjacent basic feasible solutions.
2. Recognize optimality, unboundedness, infeasibility, and alternative optima from computational evidence.
3. Use weak and strong duality to build an optimality certificate.
4. Recover and interpret shadow prices from a final simplex basis.
5. Test the local validity of a shadow-price prediction.
6. Use sensitivity information carefully in a managerial decision.

### Programming expectations

You are **not** expected to write a simplex solver. Instructor-provided cells perform tableau arithmetic, matrix calculations, solver calls, and plots. You will enter three pivot choices, run experiments, complete tables, and explain the operations-research meaning.

## Submission and grading

Submit one ZIP containing:

- `lastname_firstname_PA1.ipynb`, executed from top to bottom with outputs visible;
- `lastname_firstname_PA1.html`, exported from the same executed notebook.

| Problem | Topic | Points |
|---|---|---:|
| 1 | Simplex iterations | 20 |
| 2 | Simplex termination | 15 |
| 3 | Weak and strong duality | 15 |
| 4 | Tableau dual values and shadow prices | 20 |
| 5 | RHS sensitivity | 15 |
| 6 | Capacity decision | 15 |
| **Total** |  | **100** |

Before submitting, use **Restart Kernel and Run All Cells**, confirm that the final computation check passes, save, and export to HTML.

# The running case study: Lakeside Furnishings

Lakeside Furnishings produces three products during a one-week planning period.

| Decision variable | Meaning | Unit profit |
|---|---|---:|
| $x_1$ | Number of standard tables produced per week | 40 dollars per table |
| $x_2$ | Number of bookshelves produced per week | 30 dollars per bookshelf |
| $x_3$ | Number of metal cabinets produced per week | 50 dollars per cabinet |

The production LP is

$$\max z=40x_1+30x_2+50x_3$$

subject to

$$2x_1+x_2+3x_3\leq100 \qquad\text{(machine hours)},$$

$$x_1+2x_2+2x_3\leq80 \qquad\text{(labor hours)},$$

$$x_1+x_2\leq40 \qquad\text{(hardwood panels)},$$

$$x_1,x_2,x_3\geq0.$$

Use $s_1,s_2,s_3$ as the slack variables for machine hours, labor hours, and hardwood panels.

## Setup

Run the next two cells. They load the data and define the instructor-provided computational tools used throughout the notebook.

In [1]:
# RUN — instructor-provided setup. Do not edit.
from pathlib import Path
import json
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

DATA_DIR = Path("data")
required_files = [
    "lakeside_products.csv",
    "lakeside_resources.csv",
    "lakeside_usage.csv",
    "mystery_models.json",
    "candidate_solutions.csv",
    "capacity_options.csv",
]
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
assert not missing, f"Missing data files: {missing}. Keep the notebook beside the data folder."

products = pd.read_csv(DATA_DIR / "lakeside_products.csv")
resources = pd.read_csv(DATA_DIR / "lakeside_resources.csv")
usage = pd.read_csv(DATA_DIR / "lakeside_usage.csv")
candidates = pd.read_csv(DATA_DIR / "candidate_solutions.csv")
options = pd.read_csv(DATA_DIR / "capacity_options.csv")
mysteries = json.loads((DATA_DIR / "mystery_models.json").read_text(encoding="utf-8"))

product_ids = products["product_id"].to_list()
resource_ids = resources["resource_id"].to_list()
product_pos = {name: i for i, name in enumerate(product_ids)}
resource_pos = {name: i for i, name in enumerate(resource_ids)}

c = products["unit_profit"].to_numpy(dtype=float)
b = resources["capacity"].to_numpy(dtype=float)
A = np.zeros((len(resources), len(products)), dtype=float)
for row in usage.itertuples(index=False):
    A[resource_pos[row.resource_id], product_pos[row.product_id]] = row.units_per_product

print("Setup complete.")
print("Products:")
print(products.to_string(index=False))
print("\nResources:")
print(resources.to_string(index=False))
print("\nConstraint matrix A:")
print(pd.DataFrame(A, index=resource_ids, columns=product_ids).to_string())

Setup complete.
Products:
product_id   product_name  unit_profit
        x1 Standard table           40
        x2      Bookshelf           30
        x3  Metal cabinet           50

Resources:
resource_id   resource_name        unit  capacity
         R1   Machine hours  hours/week       100
         R2     Labor hours  hours/week        80
         R3 Hardwood panels panels/week        40

Constraint matrix A:
     x1   x2   x3
R1  2.0  1.0  3.0
R2  1.0  2.0  2.0
R3  1.0  1.0  0.0


In [2]:
# RUN — instructor-provided simplex tools. Do not edit.
TOL = 1e-9


def make_tableau(objective, matrix, rhs):
    objective = np.asarray(objective, dtype=float)
    matrix = np.asarray(matrix, dtype=float)
    rhs = np.asarray(rhs, dtype=float)
    m, n = matrix.shape
    tableau = np.zeros((m + 1, n + m + 1), dtype=float)
    tableau[:m, :n] = matrix
    tableau[:m, n:n + m] = np.eye(m)
    tableau[:m, -1] = rhs
    tableau[-1, :n] = -objective
    names = [f"x{i + 1}" for i in range(n)] + [f"s{i + 1}" for i in range(m)]
    basis = [n + i for i in range(m)]
    return tableau, basis, names


def tableau_frame(tableau, basis, names):
    """Display the objective row first while retaining the internal row layout."""
    displayed = np.vstack([tableau[-1, :], tableau[:-1, :]])
    row_names = ["z"] + [names[index] for index in basis]
    frame = pd.DataFrame(displayed, index=row_names, columns=names + ["RHS"])
    frame.insert(0, "z", [1.0] + [0.0] * len(basis))
    frame.index.name = "basic vars"
    return frame


def pivot_information(tableau, basis, names):
    entering_candidates = [j for j in range(len(names)) if tableau[-1, j] < -TOL]
    if not entering_candidates:
        return {"status": "optimal"}
    entering_index = min(entering_candidates, key=lambda j: tableau[-1, j])
    ratios = []
    for i in range(len(basis)):
        coefficient = tableau[i, entering_index]
        if coefficient > TOL:
            ratios.append({
                "row": i,
                "leaving": names[basis[i]],
                "coefficient": coefficient,
                "rhs": tableau[i, -1],
                "ratio": tableau[i, -1] / coefficient,
            })
    if not ratios:
        return {
            "status": "unbounded",
            "entering": names[entering_index],
            "entering_index": entering_index,
            "ratios": [],
        }
    leaving = min(ratios, key=lambda item: (item["ratio"], item["row"]))
    return {
        "status": "pivot",
        "entering": names[entering_index],
        "entering_index": entering_index,
        "leaving": leaving["leaving"],
        "leaving_row": leaving["row"],
        "ratios": ratios,
    }


def perform_pivot(tableau, basis, entering_index, leaving_row):
    new_tableau = tableau.copy()
    new_basis = basis.copy()
    pivot_value = new_tableau[leaving_row, entering_index]
    new_tableau[leaving_row, :] /= pivot_value
    for i in range(new_tableau.shape[0]):
        if i != leaving_row:
            new_tableau[i, :] -= new_tableau[i, entering_index] * new_tableau[leaving_row, :]
    new_basis[leaving_row] = entering_index
    new_tableau[np.abs(new_tableau) < TOL] = 0.0
    return new_tableau, new_basis


def show_tableau_for_entering_choice(tableau, basis, names):
    print(tableau_frame(tableau, basis, names).round(6).to_string())
    info = pivot_information(tableau, basis, names)
    if info["status"] == "pivot":
        print("\nThe objective row is shown first. Choose the entering variable using the most-negative-coefficient rule.")
    elif info["status"] == "optimal":
        print("\nNo negative coefficient remains in the top objective row; the tableau is optimal.")
    else:
        print(f"\nThe improving column for {info['entering']} has no eligible leaving row.")
    return info


def check_entering_and_show_ratios(tableau, basis, names, entering_choice):
    info = pivot_information(tableau, basis, names)
    if not entering_choice:
        print("No entering variable was entered. Type a variable name in the previous cell and run this cell again.")
        return {"status": "choice_needed", "choice_is_correct": False}
    if info["status"] != "pivot":
        print(f"An entering variable is not available because the tableau status is {info['status']}.")
        return {"status": info["status"], "choice_is_correct": False}
    if entering_choice != info["entering"]:
        print(f"{entering_choice} is not the entering variable under the most-negative-coefficient rule.")
        print("Recheck the top objective row, edit the previous choice cell, and run this cell again.")
        return {"status": "incorrect_entering_choice", "choice_is_correct": False}

    ratio_rows = []
    entering_index = info["entering_index"]
    for row, basic_index in enumerate(basis):
        coefficient = tableau[row, entering_index]
        eligible = coefficient > TOL
        ratio_rows.append({
            "basic_variable": names[basic_index],
            f"{entering_choice}_coefficient": coefficient,
            "RHS": tableau[row, -1],
            "ratio_RHS/coefficient": tableau[row, -1] / coefficient if eligible else np.nan,
            "eligible": "Yes" if eligible else "No",
        })
    print(f"Entering variable accepted: {entering_choice}")
    print("\nMinimum-ratio calculations (only positive entering-column coefficients are eligible):")
    print(pd.DataFrame(ratio_rows).round(6).to_string(index=False, na_rep="excluded"))
    print("\nNow choose the leaving variable from the smallest eligible nonnegative ratio.")
    info["choice_is_correct"] = True
    return info


def check_leaving_and_pivot(tableau, basis, names, entering_choice, leaving_choice):
    info = pivot_information(tableau, basis, names)
    if not leaving_choice:
        print("No leaving variable was entered. Type a variable name in the previous cell and run this cell again.")
        return tableau, basis, False
    if info["status"] != "pivot":
        print(f"A pivot is not available because the tableau status is {info['status']}.")
        return tableau, basis, False
    if entering_choice != info["entering"]:
        print("The entering-variable choice is not correct, so no pivot was performed.")
        print("Correct the entering variable and rerun the ratio cell before choosing a leaving variable.")
        return tableau, basis, False
    if leaving_choice != info["leaving"]:
        print(f"{leaving_choice} is not the leaving variable under the minimum-ratio rule.")
        print("Recheck the displayed ratio table, edit the previous choice cell, and run this cell again.")
        return tableau, basis, False
    print(f"Leaving variable accepted: {leaving_choice}")
    new_tableau, new_basis = perform_pivot(tableau, basis, info["entering_index"], info["leaving_row"])
    return new_tableau, new_basis, True


def solution_from_tableau(tableau, basis, names, decision_count):
    values = np.zeros(len(names))
    for row, variable_index in enumerate(basis):
        values[variable_index] = tableau[row, -1]
    return values[:decision_count], tableau[-1, -1]


def simplex_solve(objective, matrix, rhs, max_iterations=50):
    tableau, basis, names = make_tableau(objective, matrix, rhs)
    history = []
    for iteration in range(max_iterations):
        info = pivot_information(tableau, basis, names)
        history.append({
            "iteration": iteration,
            "basis": [names[i] for i in basis],
            "objective": float(tableau[-1, -1]),
            "info": info,
        })
        if info["status"] == "optimal":
            x, z = solution_from_tableau(tableau, basis, names, len(objective))
            nonbasic_original = [j for j in range(len(objective)) if j not in basis]
            zero_reduced = [names[j] for j in nonbasic_original if abs(tableau[-1, j]) <= TOL]
            return {
                "status": "optimal",
                "tableau": tableau,
                "basis": basis,
                "names": names,
                "x": x,
                "objective": z,
                "history": history,
                "zero_reduced_nonbasic_original": zero_reduced,
            }
        if info["status"] == "unbounded":
            return {
                "status": "unbounded",
                "tableau": tableau,
                "basis": basis,
                "names": names,
                "history": history,
                "unbounded_evidence": info,
            }
        tableau, basis = perform_pivot(
            tableau, basis, info["entering_index"], info["leaving_row"]
        )
    raise RuntimeError("Iteration limit reached")


def solve_lp(objective, matrix, rhs):
    result = linprog(-np.asarray(objective, dtype=float), A_ub=matrix, b_ub=rhs, bounds=(0, None))
    if not result.success:
        return result, None
    return result, -float(result.fun)


print("Simplex, validation, and re-optimization tools are ready. Objective rows will be displayed first.")

Simplex, validation, and re-optimization tools are ready. Objective rows will be displayed first.


# Problem 1 — Trace the simplex method to optimality
## 20 points

The notebook starts with basis $\{s_1,s_2,s_3\}$. The **objective row is always displayed as the top row**. At each iteration:

1. Inspect the top objective row and use the most-negative-coefficient rule to select the entering variable.
2. Enter only that variable in the first **YOUR CHOICE** cell.
3. Run the next cell; the notebook will verify the entering variable and display every minimum-ratio calculation.
4. Use the smallest eligible nonnegative ratio to select the leaving variable.
5. Enter the leaving variable in the next **YOUR CHOICE** cell and run the pivot.
6. Repeat until the top objective row has no negative coefficient.

If eligible ratios tie, choose the topmost tied constraint row. Record each iteration after completing all pivots.

In [3]:
# RUN — initialize the Lakeside tableau. The objective row appears first.
current_tableau, current_basis, variable_names = make_tableau(c, A, b)
iteration_log = [{
    "iteration": 0,
    "basis": [variable_names[i] for i in current_basis],
    "objective": current_tableau[-1, -1],
}]
iteration_0_info = show_tableau_for_entering_choice(current_tableau, current_basis, variable_names)

              z    x1    x2    x3   s1   s2   s3    RHS
basic vars                                             
z           1.0 -40.0 -30.0 -50.0  0.0  0.0  0.0    0.0
s1          0.0   2.0   1.0   3.0  1.0  0.0  0.0  100.0
s2          0.0   1.0   2.0   2.0  0.0  1.0  0.0   80.0
s3          0.0   1.0   1.0   0.0  0.0  0.0  1.0   40.0

The objective row is shown first. Choose the entering variable using the most-negative-coefficient rule.


In [4]:
# YOUR CHOICE — iteration 0 entering variable.
entering_0 = "x3"

In [5]:
# RUN — check the iteration 0 entering variable and show the ratio calculations.
iteration_0_info = check_entering_and_show_ratios(
    current_tableau, current_basis, variable_names, entering_0
)

No entering variable was entered. Type a variable name in the previous cell and run this cell again.


In [6]:
# YOUR CHOICE — iteration 0 leaving variable.
leaving_0 = "s1"

In [7]:
# RUN — check the iteration 0 leaving variable, pivot, and display the next tableau.
current_tableau, current_basis, pivot_0_completed = check_leaving_and_pivot(
    current_tableau, current_basis, variable_names, entering_0, leaving_0
)
if pivot_0_completed:
    iteration_log.append({"iteration": 1, "basis": [variable_names[i] for i in current_basis], "objective": current_tableau[-1, -1]})
    iteration_1_info = show_tableau_for_entering_choice(current_tableau, current_basis, variable_names)
else:
    iteration_1_info = {"status": "waiting_for_correct_iteration_0_choices"}
    print("No pivot was performed. Correct the iteration 0 choices and rerun this cell.")

No leaving variable was entered. Type a variable name in the previous cell and run this cell again.
No pivot was performed. Correct the iteration 0 choices and rerun this cell.


In [8]:
# YOUR CHOICE — iteration 1 entering variable.
entering_1 = "x2"

In [9]:
# RUN — check the iteration 1 entering variable and show the ratio calculations.
iteration_1_info = check_entering_and_show_ratios(
    current_tableau, current_basis, variable_names, entering_1
)

No entering variable was entered. Type a variable name in the previous cell and run this cell again.


In [10]:
# YOUR CHOICE — iteration 1 leaving variable.
leaving_1 = "s2"

In [11]:
# RUN — check the iteration 1 leaving variable, pivot, and display the next tableau.
current_tableau, current_basis, pivot_1_completed = check_leaving_and_pivot(
    current_tableau, current_basis, variable_names, entering_1, leaving_1
)
if pivot_1_completed:
    iteration_log.append({"iteration": 2, "basis": [variable_names[i] for i in current_basis], "objective": current_tableau[-1, -1]})
    iteration_2_info = show_tableau_for_entering_choice(current_tableau, current_basis, variable_names)
else:
    iteration_2_info = {"status": "waiting_for_correct_iteration_1_choices"}
    print("No pivot was performed. Correct the iteration 1 choices and rerun this cell.")

No leaving variable was entered. Type a variable name in the previous cell and run this cell again.
No pivot was performed. Correct the iteration 1 choices and rerun this cell.


In [12]:
# YOUR CHOICE — iteration 2 entering variable.
entering_2 = "x1"

In [13]:
# RUN — check the iteration 2 entering variable and show the ratio calculations.
iteration_2_info = check_entering_and_show_ratios(
    current_tableau, current_basis, variable_names, entering_2
)

No entering variable was entered. Type a variable name in the previous cell and run this cell again.


In [ ]:
# YOUR CHOICE — iteration 2 leaving variable.
leaving_2 = "s3"

In [ ]:
# RUN — check the iteration 2 leaving variable, perform the pivot, and finish.
current_tableau, current_basis, pivot_2_completed = check_leaving_and_pivot(
    current_tableau, current_basis, variable_names, entering_2, leaving_2
)
if not pivot_2_completed:
    raise RuntimeError("Correct the iteration 2 choices before continuing.")
iteration_log.append({"iteration": 3, "basis": [variable_names[i] for i in current_basis], "objective": current_tableau[-1, -1]})
final_info = show_tableau_for_entering_choice(current_tableau, current_basis, variable_names)
assert final_info["status"] == "optimal"

optimal_x, optimal_z = solution_from_tableau(current_tableau, current_basis, variable_names, decision_count=3)
iteration_history = pd.DataFrame(iteration_log)
print("\nIteration history:")
print(iteration_history.to_string(index=False))
print("\nOptimal production quantities:", {name: float(value) for name, value in zip(product_ids, optimal_x.round(6))})
print(f"Maximum weekly profit: ${optimal_z:,.2f}")
print("Resource use:", {name: float(value) for name, value in zip(resource_ids, (A @ optimal_x).round(6))})
print("Resource slack:", {name: float(value) for name, value in zip(resource_ids, (b - A @ optimal_x).round(6))})

### YOUR RESPONSE 1A — iteration record

For each iteration, use the tableau **before the pivot**. List the current basic variables in constraint-row order, the entering variable selected from the top objective row, every eligible ratio shown by the notebook, the leaving variable selected by the minimum-ratio rule, and the current objective value. In the final row, report the final basis and optimal objective value; there is no entering or leaving variable because no further pivot is needed.

| Iteration | Current basis | Entering variable | Vars with eligible ratios | Leaving variable | Objective before pivot |
|---:|---|---|---|---|---:|
| 0 | s1, s2, s3 | x3 | s1 (33.33), s2 (40) | s1 | 0 |
| 1 | x3, s2, s3 | x2 | x3 (100), s2 (10), s3 (40) | s2 | 1666.67 |
| 2 | x3, x2, s3 | x1 | x3 (40), s3 (24) | s3 | 1800 |
| Final | x1, x2, x3 | None | None | None | 2040 |

### YOUR RESPONSE 1B — simplex interpretation

The top objective row holds the negative of each variable's reduced cost, so the most negative entry marks the nonbasic variable that raises profit fastest per unit brought into the basis; at iteration 0 that was x3 at -50, ahead of x1 at -40 and x2 at -30. A constraint row is eligible for the ratio test only when its entry in the entering column is strictly positive, because that is the only case in which increasing the entering variable drives that row's basic variable down toward zero: a zero entry leaves the basic variable untouched, and a negative entry makes it grow, so neither can ever be the binding limit. The smallest eligible ratio is therefore the largest value the entering variable can take before some basic variable would turn negative, and the variable sitting in that row is the one that leaves. Optimality is established when no entry in the top objective row is negative, meaning no remaining variable can be increased to improve z; that condition was reached after three pivots at basis {x1, x2, x3}. The optimal production plan is 24 standard tables, 16 bookshelves, and 12 metal cabinets per week, for a maximum weekly profit of $2,040. All three constraints are binding, since machine hours use 100 of 100, labor hours 80 of 80, and hardwood panels 40 of 40, leaving every slack variable at zero and no resource idle.

# Problem 2 — Diagnose simplex termination conditions
## 15 points

Run the diagnostic cells and classify each mystery LP as having a unique finite optimum, being unbounded, or being infeasible. Use specific tableau or initialization evidence—not only the classification.

In [ ]:
# RUN — Mystery A diagnostic.
case_a = mysteries["A"]
result_a = simplex_solve(case_a["c"], case_a["A"], case_a["b"])
print("Mystery A final simplex tableau:")
print(tableau_frame(result_a["tableau"], result_a["basis"], result_a["names"]).round(6).to_string())
print("Nonbasic original variables with zero reduced cost:", result_a["zero_reduced_nonbasic_original"])

In [ ]:
# RUN — Mystery B diagnostic.
case_b = mysteries["B"]
result_b = simplex_solve(case_b["c"], case_b["A"], case_b["b"])
print("Mystery B last tableau before termination:")
print(tableau_frame(result_b["tableau"], result_b["basis"], result_b["names"]).round(6).to_string())
print("Entering candidate:", result_b["unbounded_evidence"]["entering"])
print("Eligible positive pivot-column entries:", result_b["unbounded_evidence"]["ratios"])

## Mystery C

Consider the LP

$$\max z=x_1$$

subject to

$$x_1\leq1,$$

$$x_1\geq2,$$

$$x_1\geq0.$$

Follow the course initialization convention:

1. Write every constraint in $\leq$ form, giving $x_1\leq1$ and $-x_1\leq-2$.
2. Add one slack variable to each constraint: $x_1+s_1=1$ and $-x_1+s_2=-2$.
3. Because the second right-hand side is negative, introduce a new initialization slack variable $s_3$ for that row and subtract $Ms_3$ in the maximization objective.
4. Multiply the initialized second equation by $-1$ so that $s_3$ is basic: $x_1-s_2+s_3=2$.

Run the next cell to inspect the initialized and final Big-$M$ tableaux. Use the fact that $M$ is a sufficiently large positive number.

In [ ]:
# RUN — Mystery C: initialization with a negative RHS and a Big-M penalty.
case_c = mysteries["C"]
print("Mystery C original problem: MAXIMIZE z = x1")
print("subject to x1 <= 1, x1 >= 2, and x1 >= 0")
print("\nStandard form with <= constraints:")
print("x1 <= 1")
print("-x1 <= -2")
print("\nAfter adding ordinary slack variables:")
print("x1 + s1 = 1")
print("-x1 + s2 = -2")
print("\nInitialization of the negative-RHS row:")
print("Introduce s3, subtract M*s3 in the objective, and use x1 - s2 + s3 = 2.")

initialized_big_m_tableau = pd.DataFrame(
    [
        [1, "-(M+1)", 0, "M", 0, "-2M"],
        [0, 1, 1, 0, 0, 1],
        [0, 1, 0, -1, 1, 2],
    ],
    index=["z", "s1", "s3"],
    columns=["z", "x1", "s1", "s2", "s3", "RHS"],
)
initialized_big_m_tableau.index.name = "basic vars"

final_big_m_tableau = pd.DataFrame(
    [
        [1, 0, "M+1", "M", 0, "1-M"],
        [0, 1, 1, 0, 0, 1],
        [0, 0, -1, -1, 1, 1],
    ],
    index=["z", "x1", "s3"],
    columns=["z", "x1", "s1", "s2", "s3", "RHS"],
)
final_big_m_tableau.index.name = "basic vars"

initialization_slack_value = 1.0
print("\nInitialized Big-M tableau (objective row first):")
print(initialized_big_m_tableau.to_string())
print("\nFinal Big-M tableau after x1 enters and s1 leaves:")
print(final_big_m_tableau.to_string())
print(f"\nInitialization slack s3 remains basic with value {initialization_slack_value:.6f}.")

## Questions for Problem 2

1. Classify Mystery A, Mystery B, and Mystery C as having a unique finite optimum, being unbounded, or being infeasible.
2. For each classification, cite specific evidence from the displayed tableau or initialization output.

### YOUR RESPONSE 2A — classifications and evidence

| Mystery LP | Classification | Specific tableau or initialization evidence |
|---|---|---|
| A | Unique finite optimum | The top objective row has no negative entry (s1 = 2, s2 = 1, s3 = 0), so the tableau is optimal at z = 10 with x1 = 2 and x2 = 2. The reported list of nonbasic original variables with zero reduced cost is empty, so no alternative optimal vertex exists. |
| B | Unbounded | x2 still carries a negative objective-row entry of -2 and is named as the entering candidate, but the list of eligible positive pivot-column entries is empty; the only constraint row holds -1 in the x2 column. No row limits the increase of x2. |
| C | Infeasible | In the final Big-M tableau the initialization slack s3 is still basic at the strictly positive value 1, and the objective value stands at 1 - M, which falls without bound as M grows. |

### YOUR RESPONSE 2B — explanation

Mystery A terminates at a genuine optimum because every entry in its top objective row is nonnegative, so no nonbasic variable can be increased to raise z above 10, and because the set of nonbasic original variables with zero reduced cost is empty, the optimal vertex is unique rather than one endpoint of a tied edge of alternative optima. Mystery B passes the entering test but fails the ratio test: x2 has a negative objective-row coefficient, yet its column contains no strictly positive entry, so increasing x2 never forces any basic variable down to zero and z can be raised indefinitely. A negative objective-row coefficient combined with an empty eligible-ratio list is the standard tableau signature of an unbounded LP, and it is a property of the feasible region rather than an arithmetic failure. Mystery C is infeasible because x1 <= 1 and x1 >= 2 cannot both hold, and the Big-M initialization detects exactly this: the initialization slack s3 introduced for the negative right-hand-side row remains basic at value 1 instead of being driven to zero. Since s3 is penalized by -M in the maximization objective, the terminal objective value 1 - M is arbitrarily negative for sufficiently large M, which is the Big-M method's way of reporting that the original constraint set contains no feasible point.

# Problem 3 — Build a primal–dual optimality certificate
## 15 points

The Lakeside **primal problem is a maximization problem**:

$$\max\;40x_1+30x_2+50x_3\qquad\text{subject to }Ax\leq b,\;x\geq0.$$

Its dual is the minimization problem

$$\min 100y_1+80y_2+40y_3$$

subject to

$$2y_1+y_2+y_3\geq40,$$
$$y_1+2y_2+y_3\geq30,$$
$$3y_1+2y_2\geq50,$$
$$y_1,y_2,y_3\geq0.$$

The data file supplies three primal candidate points P1–P3 and three dual candidate points D1–D3. Run the next cell to check feasibility, objective values, and constraint violations. You must then determine whether each candidate point supplies a valid lower bound, a valid upper bound, or neither.

In [ ]:
# RUN — check the supplied primal and dual candidate points.
def check_primal(x):
    x = np.asarray(x, dtype=float)
    use = A @ x
    violations = np.maximum(use - b, 0)
    nonneg_violation = np.maximum(-x, 0)
    feasible = max(violations.max(), nonneg_violation.max()) <= TOL
    return feasible, float(c @ x), use, violations


def check_dual(y):
    y = np.asarray(y, dtype=float)
    lhs = A.T @ y
    violations = np.maximum(c - lhs, 0)
    nonneg_violation = np.maximum(-y, 0)
    feasible = max(violations.max(), nonneg_violation.max()) <= TOL
    return feasible, float(b @ y), lhs, violations


point_records = []
for row in candidates.itertuples(index=False):
    values = np.array([row.value_1, row.value_2, row.value_3], dtype=float)
    if row.type == "primal":
        feasible, objective, lhs, violations = check_primal(values)
        bound_type = "lower" if feasible else "neither"
    else:
        feasible, objective, lhs, violations = check_dual(values)
        bound_type = "upper" if feasible else "neither"
    point_records.append({
        "candidate_point": row.candidate_point,
        "problem_side": row.type,
        "point": values.tolist(),
        "feasible": feasible,
        "objective_value": objective,
        "largest_constraint_violation": float(violations.max()),
        "bound_type": bound_type,
    })

candidate_point_results = pd.DataFrame(point_records)
print("Primal problem type: MAXIMIZATION")
print("Dual problem type: MINIMIZATION")
print("\nCandidate-point feasibility and objective results:")
print(candidate_point_results.drop(columns=["bound_type"]).to_string(index=False))

feasible_primal = candidate_point_results.query("problem_side == 'primal' and feasible == True")
feasible_dual = candidate_point_results.query("problem_side == 'dual' and feasible == True")
best_lower_bound = feasible_primal["objective_value"].max()
best_upper_bound = feasible_dual["objective_value"].min()
print(f"\nBest feasible-primal objective: {best_lower_bound:,.6f}")
print(f"Best feasible-dual objective: {best_upper_bound:,.6f}")

### YOUR RESPONSE 3A — feasibility and objective values

| Candidate point | Primal or dual? | Point | Feasible? | Objective value | Violated constraint, if any |
|---|---|---|---|---:|---|
| P1 | Primal | (20, 10, 10) | Yes | 1600 | None; usage (80, 60, 30) is within (100, 80, 40) |
| P2 | Primal | (25, 20, 15) | No | 2350 | All three: machine hours 115 > 100 (by 15), labor hours 95 > 80 (by 15), hardwood panels 45 > 40 (by 5) |
| P3 | Primal | (24, 16, 12) | Yes | 2040 | None; usage (100, 80, 40) exactly meets capacity |
| D1 | Dual | (15, 5, 10) | Yes | 2300 | None; dual left sides (45, 35, 55) clear (40, 30, 50) |
| D2 | Dual | (10, 4, 8) | No | 1640 | All three dual constraints: the x3 constraint 38 < 50 (short by 12), the x1 constraint 32 < 40 (short by 8), the x2 constraint 26 < 30 (short by 4) |
| D3 | Dual | (14, 4, 8) | Yes | 2040 | None; dual left sides (40, 30, 50) meet (40, 30, 50) with equality |

### YOUR RESPONSE 3B — bounds supplied by candidate points

| Candidate point | Lower bound, upper bound, or neither? | Bound value, if valid | Reason |
|---|---|---:|---|
| P1 | Lower bound | 1600 | A feasible primal point is an achievable production plan, so its objective can never exceed the primal maximum. |
| P2 | Neither | — | The point is infeasible, so its objective value of 2350 describes no achievable plan and bounds nothing. |
| P3 | Lower bound | 2040 | A feasible primal point, and the strongest lower bound of the three. |
| D1 | Upper bound | 2300 | Weak duality states that any feasible dual objective is at least the primal maximum. |
| D2 | Neither | — | The point is dual-infeasible, so weak duality does not apply to it. |
| D3 | Upper bound | 2040 | A feasible dual point, and the tightest upper bound of the three. |

### YOUR RESPONSE 3C — optimality certificate

The best valid lower bound is 2040, supplied by the feasible primal point P3 = (24, 16, 12), and the best valid upper bound is also 2040, supplied by the feasible dual point D3 = (14, 4, 8). The pair P3 and D3 therefore provides an optimality certificate, because together they trap the primal optimum in the interval [2040, 2040], which contains the single value 2040. Weak duality guarantees that every feasible primal objective is less than or equal to every feasible dual objective, so the true optimum always lies between the best lower and best upper bound. When a feasible primal point and a feasible dual point produce the same objective value, that common value is simultaneously an attainable profit and an unbeatable ceiling, so neither point can be improved and both must be optimal. This is the content of strong duality for linear programs: the primal and dual optima are equal, and exhibiting one matching feasible pair proves optimality with no further pivoting. The certificate is also self-checking, since it requires only that each point satisfy its own constraints and that the two objective values agree, both of which can be verified by direct substitution. Note that D3 equals the vector c_B^T B^-1 = (14, 4, 8) recovered from the final basis in Problem 4, confirming that the simplex method finished at the same primal-dual pair.

# Problem 4 — Recover shadow prices from the final tableau
## 20 points

The final basis from Problem 1 is the set

$$\{x_1,x_2,x_3\}.$$

The notebook constructs the corresponding basis matrix $B$ and computes

$$y^\top=c_B^\top B^{-1}.$$

It compares these dual values with the slack-variable coefficients in the **top objective row** of the final tableau and with the actual objective changes obtained by adding one unit of each resource and re-optimizing.

In [ ]:
# RUN — recover the dual values from the final basis.
observed_final_basis = {variable_names[i] for i in current_basis}
required_final_basis = {"x1", "x2", "x3"}
assert observed_final_basis == required_final_basis, (
    f"Expected final basis {required_final_basis}, but found {observed_final_basis}."
)

final_basis_names = ["x1", "x2", "x3"]
final_original_indices = [int(name[1:]) - 1 for name in final_basis_names]
B = A[:, final_original_indices]
c_B = c[final_original_indices]
B_inverse = np.linalg.inv(B)
y_from_basis = c_B @ B_inverse

print("Final basis: {x1, x2, x3}")
print("\nBasis matrix B:")
print(pd.DataFrame(B, index=resource_ids, columns=final_basis_names).to_string())
print("\nB inverse:")
print(pd.DataFrame(B_inverse, index=final_basis_names, columns=resource_ids).round(6).to_string())
print("\nc_B^T:", c_B)
print("c_B^T B^-1:", y_from_basis)

In [ ]:
# RUN — compare the final tableau, matrix formula, and one-unit re-optimization.
tableau_shadow = current_tableau[-1, 3:6]
base_result, base_objective = solve_lp(c, A, b)

comparison_records = []
for i, resource_id in enumerate(resource_ids):
    changed_b = b.copy()
    changed_b[i] += 1
    changed_result, changed_objective = solve_lp(c, A, changed_b)
    comparison_records.append({
        "resource_id": resource_id,
        "resource_name": resources.loc[i, "resource_name"],
        "top_objective_row_value": tableau_shadow[i],
        "basis_formula_value": y_from_basis[i],
        "actual_objective_change_from_plus_1": changed_objective - base_objective,
    })

shadow_comparison = pd.DataFrame(comparison_records)
print(shadow_comparison.round(6).to_string(index=False))
print(f"\nBase objective from independent re-optimization: ${base_objective:,.6f}")

### YOUR RESPONSE 4A — comparison table

| Resource | Top objective-row value | $c_B^TB^{-1}$ | Actual objective change from +1 unit |
|---|---:|---:|---:|
| Machine hours | 14 | 14 | 14 |
| Labor hours | 4 | 4 | 4 |
| Hardwood panels | 8 | 8 | 8 |

### YOUR RESPONSE 4B — interpretation

Each shadow price is the extra weekly profit earned from one additional unit of that resource, holding everything else fixed: one more machine hour is worth $14, one more labor hour $4, and one more hardwood panel $8. Machine hours are locally the most valuable resource, so if only one unit of capacity could be added, it should be a machine hour, and management should be willing to pay anything below $14 for it. The three methods agree because they are three readings of the same quantity. The slack coefficients in the top objective row of the final tableau are exactly the reduced costs of the slack variables, which equal c_B^T B^-1 by construction, since the simplex method reaches the final tableau by multiplying the original system by B^-1. The one-unit re-optimization experiment agrees as well because adding a single unit to any right-hand side is a small enough change that the optimal basis {x1, x2, x3} stays optimal, and while the basis is unchanged the objective moves linearly at precisely the rate y_i. The final tableau and the dual solution therefore carry the same economic information in two notations: the dual variables price the scarce resources, and the tableau's objective row displays those same prices in the slack columns, which is why the dual-feasible point D3 = (14, 4, 8) from Problem 3 reappears here as c_B^T B^-1. The agreement is local rather than permanent, and Problem 5 shows how far it extends.

# Problem 5 — Test the range of a shadow-price prediction
## 15 points

The notebook varies machine capacity from 30 to 170 hours. At each value, it records the re-optimized objective, the original shadow-price prediction, prediction error, basis, production plan, and slack.

The base prediction is

$$\widehat z(b_1+\Delta)=z(b_1)+y_1\Delta.$$

In [ ]:
# RUN — machine-capacity sensitivity experiment.
machine_capacities = np.arange(30.0, 171.0, 10.0)
base_machine_capacity = b[0]
base_machine_shadow = y_from_basis[0]
sensitivity_records = []

for capacity in machine_capacities:
    changed_b = b.copy()
    changed_b[0] = capacity
    simplex_result = simplex_solve(c, A, changed_b)
    assert simplex_result["status"] == "optimal"
    actual = simplex_result["objective"]
    predicted = optimal_z + base_machine_shadow * (capacity - base_machine_capacity)
    sensitivity_records.append({
        "machine_capacity": capacity,
        "actual_profit": actual,
        "shadow_price_prediction": predicted,
        "absolute_error": abs(actual - predicted),
        "basis": ", ".join(simplex_result["names"][i] for i in simplex_result["basis"]),
        "x1": simplex_result["x"][0],
        "x2": simplex_result["x"][1],
        "x3": simplex_result["x"][2],
        "machine_slack": changed_b[0] - A[0] @ simplex_result["x"],
    })

sensitivity = pd.DataFrame(sensitivity_records)
print(sensitivity.round(6).to_string(index=False))

plt.figure(figsize=(8, 5))
plt.plot(sensitivity["machine_capacity"], sensitivity["actual_profit"], marker="o", label="Actual re-optimized profit")
plt.plot(sensitivity["machine_capacity"], sensitivity["shadow_price_prediction"], linestyle="--", label="Original shadow-price prediction")
plt.axvline(base_machine_capacity, color="gray", linewidth=1)
plt.xlabel("Available machine hours")
plt.ylabel("Maximum weekly profit ($)")
plt.title("Machine-hour sensitivity")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

### YOUR RESPONSE 5 — sensitivity

The original prediction is exact within numerical tolerance over the machine-capacity interval from 40 to 140 hours, where the absolute error is 0 at every tested value. The first tested value below that interval is 30 hours, where the error is $160, and the first tested value above it is 150 hours, where the error is $140. The production plan changes at both breakpoints. Below 40 hours the plan collapses to x = (0, 30, 0) on basis {x2, s2, s3}, dropping both tables and cabinets, while within the exact interval the basis is {x1, x2, x3} and the plan slides smoothly from (0, 40, 0) at 40 hours to (40, 0, 20) at 140 hours, trading bookshelves for tables and cabinets as machine time is added. Above 140 hours the basis becomes {s1, x3, x1}, the plan freezes at (40, 0, 20), and s1 enters the basis as genuine idle machine time, growing to 10, 20, and 30 hours at capacities of 150, 160, and 170. When 60 additional machine hours are supplied, capacity reaches 160, the prediction is 2040 + 14(60) = $2,880, the actual re-optimized profit is $2,600, and the prediction error is $280. Additional machine capacity eventually stops increasing profit because machine hours are no longer the binding restriction: at 140 hours labor and hardwood panels are both exhausted, so beyond that point extra machine time cannot be converted into extra output and the plot's solid actual-profit line flattens while the dashed prediction keeps climbing. The machine-hour shadow price consequently falls to $0 after sufficiently large increases, which is the slope of that flat segment. A shadow price is local rather than permanent because it is the rate of improvement associated with one particular optimal basis, and it is valid only over the right-hand-side range for which that basis stays both feasible and optimal; once a basis change occurs at a breakpoint, a new shadow price takes over and any linear extrapolation built on the old one overstates the benefit.

# Problem 6 — Recommend a one-week capacity rental
## 15 points

Management may rent one capacity package for the next planning week. Rental costs and operating profits refer to the same one-week period.

First screen each option using the original shadow prices. Then re-solve the LP and compare actual net benefits.

In [ ]:
# RUN — shadow-price screening and full re-optimization.
option_records = []
for row in options.itertuples(index=False):
    delta = np.array([row.delta_machine, row.delta_labor, row.delta_hardwood], dtype=float)
    predicted_gross = float(y_from_basis @ delta)
    predicted_net = predicted_gross - row.rental_cost
    changed_b = b + delta
    reoptimized = simplex_solve(c, A, changed_b)
    assert reoptimized["status"] == "optimal"
    actual_gross = reoptimized["objective"] - optimal_z
    actual_net = actual_gross - row.rental_cost
    changed_basis_names = [reoptimized["names"][i] for i in reoptimized["basis"]]
    option_records.append({
        "option": row.option,
        "description": row.description,
        "rental_cost": row.rental_cost,
        "predicted_gross_benefit": predicted_gross,
        "predicted_net_benefit": predicted_net,
        "actual_gross_benefit": actual_gross,
        "actual_net_benefit": actual_net,
        "final_basis": ", ".join(changed_basis_names),
        "basis_changed": set(changed_basis_names) != set(final_basis_names),
    })

option_results = pd.DataFrame(option_records)
print(option_results.round(6).to_string(index=False))
print("\nBest option by predicted net benefit:", option_results.loc[option_results["predicted_net_benefit"].idxmax(), "option"])
print("Best option by actual net benefit:", option_results.loc[option_results["actual_net_benefit"].idxmax(), "option"])

### YOUR RESPONSE 6A — decision table

| Option | Predicted gross benefit | Predicted net benefit | Actual gross benefit | Actual net benefit | Basis changed? |
|---|---:|---:|---:|---:|---|
| A | 280 | 80 | 280 | 80 | No |
| B | 120 | 40 | 120 | 40 | No |
| C | 240 | 130 | 160 | 50 | Yes |
| D | 840 | 140 | 560 | -140 | Yes |
| None | 0 | 0 | 0 | 0 | No |

### YOUR RESPONSE 6B — decision memo

**To:** Operations Manager
**Subject:** One-week capacity rental recommendation

I recommend renting Option A, twenty additional machine hours for $200. Shadow-price screening predicted a gross benefit of $280 and a net benefit of $80, and full re-optimization confirmed both exactly. Option A is the only package whose predicted and actual results agree while leaving the optimal basis {x1, x2, x3} unchanged, so its return does not depend on restructuring the production plan.

Screening and re-optimization did not produce the same ranking. By predicted net benefit the order was D ($140), C ($130), A ($80), B ($40). By actual net benefit it was A ($80), C ($50), B ($40), and D at a loss of $140. Option D is the misleading prediction. Its screening value assumed the $14 machine-hour shadow price applies to all sixty added hours, but that price holds only while machine capacity stays at or below 140 hours. Option D raises capacity to 160, past the point where labor and hardwood become binding, so the final twenty hours earn nothing. Actual gross benefit is $560 rather than $840, turning a predicted $140 gain into a $140 loss.

Two limitations: the analysis covers one week and excludes setup, training, or contract costs beyond the quoted rental, and it assumes every additional unit sells at current unit profits, treating demand as unlimited.

Next step: compute the exact right-hand-side range for each resource before negotiating, so every quoted rental can be screened against the interval where its shadow price is valid.

# Final computation check

Run this after completing the notebook. It checks the required calculations but cannot grade your written explanations.

In [ ]:
# RUN — final computation check. Do not edit.
assert final_info["status"] == "optimal"
assert np.max(A @ optimal_x - b) <= 1e-7
assert abs(optimal_z - base_objective) <= 1e-7
assert abs(best_lower_bound - best_upper_bound) <= 1e-7
assert np.max(np.abs(tableau_shadow - y_from_basis)) <= 1e-7
assert len(sensitivity) == 15
assert len(option_results) == 5
assert initialization_slack_value > TOL

print("COMPUTATION CHECK PASSED")
print("Confirm that every YOUR RESPONSE prompt contains your work.")
print("Restart Kernel and Run All Cells, save, export to HTML, and submit both files in one ZIP.")